In [1]:
import numpy as np
import matplotlib.pyplot as plt
from galpy.potential import (NFWPotential, TwoPowerSphericalPotential, AnySphericalPotential,evaluateDensities)
from colossus.cosmology import cosmology
from colossus.halo import concentration, profile_nfw
from scipy.optimize import minimize, fsolve
from joblib import Parallel, delayed
from math import pi
from galpy.orbit import Orbit       
import sys
import pandas as pd
import time
from astropy.constants import G
from astropy import units as u
from galpy.potential import ChandrasekharDynamicalFrictionForce as CDFF, vcirc, FDMDynamicalFrictionForce as FDM_DF
from joblib import Parallel, delayed
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from halotools.sim_manager import FakeSim
from halotools.empirical_models import PrebuiltSubhaloModelFactory
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde
from matplotlib.ticker import AutoMinorLocator
from matplotlib.ticker import MultipleLocator
from scipy.signal import argrelextrema
import random
cosmology.setCosmology('planck15')

In [2]:

def zeta(z, Omega_m):
    return (18 * np.pi**2 + 82 * (Omega_m - 1) - 39 * (Omega_m - 1)**2) / Omega_m
def r_c(z, M_h, m_22, Omega_m):
    return (1.6 / np.sqrt(1 + z)) * (zeta(z, Omega_m) / zeta(0, Omega_m0))**(-1/6) * (M_h / 1e9)**(-1/3) * m_22**-1
def rho2_z(r_c, m_22, z):
    return 1.9 * (10 * m_22)**-2 * r_c**-4 * 1e9 / ((1 + z)**3)

zgo = 0
cosmo = cosmology.setCosmology('planck18')
Omega_m0 = cosmo.Om(zgo)

def FDMhalo(m22, M):

    Mh=M*0.704
    rc1 = r_c(zgo, M, m22, Omega_m0)
    rhoFDM = rho2_z(rc1, m22, zgo)
    AmpFDM = rhoFDM * 4 * pi * rc1**3

    rc2 = r_c(zgo, M, m22/10, Omega_m0)
    rhoFDM2 = rho2_z(rc2, m22/10, zgo)
    AmpFDM2 = rhoFDM2 * 4 * pi * rc2**3
    
    C2 = concentration.concentration(M*0.704, 'vir', zgo, model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=M*0.704, c=C2, z=zgo, mdef='vir')
    rs = p_nfw.par['rs'] / 0.704
    A=np.log(1+C2)-C2/(1+C2)

    DMhalo2 = NFWPotential(amp=M/A * u.Msun, a=rs*u.kpc)
    r_vals = np.logspace(np.log10(0.001), np.log10(1000), 200)
    FDMhalo_soliton = AnySphericalPotential(dens=lambda r: rhoFDM / ((1 + 0.091 * (r / rc1)**2)**8))

    def cost(params):
        psi, gamma, eta = params
        if psi <= 0 or gamma <= 0 or eta <= 0:
            return np.inf

        def equation(alpha):
            FDMhalo3 = TwoPowerSphericalPotential(amp=psi * AmpFDM2 * (alpha**3) * u.Msun, a=alpha * rc2 * u.kpc, alpha=0, beta=3)
            dens_FDM3 = evaluateDensities(FDMhalo3, 100 * u.kpc, 0 * u.kpc)
            dens_CDM = evaluateDensities(DMhalo2, 100 * u.kpc, 0 * u.kpc)
            return (dens_FDM3 - dens_CDM) / dens_CDM

        try:
            alpha_sol = fsolve(equation, 10)[0]
        except:
            return np.inf

        FDMhalo3 = TwoPowerSphericalPotential(amp=psi * AmpFDM2 * (alpha_sol**3) * u.Msun, a=alpha_sol * rc2 * u.kpc, alpha=0, beta=3)
        CoreFDM = TwoPowerSphericalPotential(amp=AmpFDM * (gamma**3) * u.Msun, a=gamma * rc1 * u.kpc, alpha=0, beta=eta)
        total_ref = np.array([evaluateDensities(FDMhalo_soliton, r, 0) + evaluateDensities(FDMhalo3, r * u.kpc, 0 * u.kpc) * 1e9 for r in r_vals])
        total_test = np.array([evaluateDensities(CoreFDM, r * u.kpc, 0 * u.kpc) * 1e9 + evaluateDensities(FDMhalo3, r * u.kpc, 0 * u.kpc) * 1e9 for r in r_vals])
        return np.mean((np.log10(total_ref + 1e-30) - np.log10(total_test + 1e-30))**2)

    result = minimize(cost, x0=[0.5, 12.0, 20.0], bounds=[(0.0001, 0.7 * (1e10 / M) * (10 / m22)), (1, 10), (3, 10)], method='L-BFGS-B')
    psi_opt, gamma_opt, eta_opt = result.x

    def equation(alpha):
        FDMhalo3 = TwoPowerSphericalPotential(amp=psi_opt * AmpFDM2 * (alpha**3) * u.Msun, a=alpha * rc2 * u.kpc, alpha=0, beta=3)
        dens_FDM3 = evaluateDensities(FDMhalo3, 100 * u.kpc, 0 * u.kpc) * 1e9
        dens_CDM = evaluateDensities(DMhalo2, 100 * u.kpc, 0 * u.kpc) * 1e9
        return (dens_FDM3 - dens_CDM) / dens_CDM

    alpha_final = fsolve(equation, 10)[0]
    FDMhalo3 = TwoPowerSphericalPotential(amp=psi_opt * AmpFDM2 * (alpha_final**3) * u.Msun, a=alpha_final * rc2 * u.kpc, alpha=0, beta=3)
    CoreFDM = TwoPowerSphericalPotential(amp=(gamma_opt**3) * AmpFDM * u.Msun, a=gamma_opt * rc1 * u.kpc, alpha=0, beta=eta_opt)
    return CoreFDM + FDMhalo3



        #tri des données euclid

def mass_from_mag(MAG_GC):
    return 1.8 * 10**(-0.4 * (MAG_GC + 0.5 - 4.83))

def count_unique(arr):
    return len(set(arr))

Rlimit=0.2 #kpc
MAGlimit = -8

file_path = "ERO_Perseus_GCs_results_for_Pierre_Feb2026.csv"
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

#logMs=np.log10(df["logMstar_host"].to_numpy())   #fornax
logMs=df["logMstar_host"].to_numpy()         #perseus
IDgal=df["gal_name"].to_numpy()
MAG_GC=df["GC_VIS_abs"].to_numpy()
RGC=df["GC_dist_kpc"].to_numpy()
print("total number of GCs :",len(RGC))
print('total number of galaxies :',count_unique(IDgal))


sola=np.where(MAG_GC<=MAGlimit)[0]
logMs=logMs[sola]
RGC=RGC[sola]
MAG_GC=MAG_GC[sola]
IDgal=IDgal[sola]

sol=np.where(RGC>Rlimit)[0]
logMs=logMs[sol]
MAG_GC=MAG_GC[sol]
RGC=RGC[sol]
IDgal=IDgal[sol]
MGC=mass_from_mag(MAG_GC)

print("final number of GCs ",len(RGC))
print('final number of galaxies (contening GCs):',count_unique(IDgal))
df_clean = pd.DataFrame({
    "galaxy": IDgal,
    "R": RGC,
    "MAG": MAG_GC,
    "logMs": logMs})

groups = df_clean.groupby("galaxy")

total number of GCs : 6827
total number of galaxies : 857
final number of GCs  4487
final number of galaxies (contening GCs): 749


In [3]:
#test tous les m22

ts = np.linspace(0,10,1001)*u.Gyr
rhm = 0.01*u.kpc
halocat = FakeSim()
model = PrebuiltSubhaloModelFactory('behroozi10')
model.populate_mock(halocat)

df = pd.DataFrame({"ID":IDgal,"R":RGC,"MAG":MAG_GC,"logMs":logMs})
groups = df.groupby("ID")

m22_values = np.arange(0.1,100.0001,0.05)
m22_c2D = {}
f = open("m22_Perseus_test.txt","a")

def test_local(i,m22,R,M,Mg,NFW_pot):
    pot = FDMhalo(m22,Mg)
    orb = Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
    fdf = FDM_DF(GMs=M[i]*u.Msun,rhm=rhm,dens=pot,m=(m22*1e-22)*u.eV)
    orb.integrate(ts,pot+fdf,method='dop853_c')
    return np.min(orb.r(ts)) < Rlimit

def find_m22(i,R,M,Mg,NFW_pot):
    res = np.array([test_local(i,m22,R,M,Mg,NFW_pot) for m22 in m22_values])
    v = m22_values[~res]
    return np.NAN if len(v)==0 else v[-1]

for name,g in groups:
    R = g["R"].to_numpy()
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)
    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg*0.704
    c = concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs = p_nfw.par['rs']/0.704
    A = np.log(1+c)-c/(1+c)
    NFW_pot = NFWPotential(amp=Mg/A*u.Msun,a=rs*u.kpc)
    os = [[r*u.kpc,0*u.km/u.s,vcirc(NFW_pot,r*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg] for r in R]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun,rhm=rhm,dens=NFW_pot)
        orb.integrate(ts,NFW_pot+cdf,method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]
    os = [os[i] for i in valid]

    vals = [find_m22(i,R,M,Mg,NFW_pot) for i in range(len(R))]
    print(name,vals)
    if len(vals) > 0:
        m22_c2D[name] = np.min(vals)
        f.write(f"{name},{np.min(vals)}\n")
        f.flush()
f.close()

mc2D = np.array(list(m22_c2D.values()))
np.savez("Perseus_test_mc2d.npz", mc2D=mc2D)

"""
m22_c3D = {}
for name, g in groups:
    R = g["R"].to_numpy()*2/np.sqrt(3)
    N_amas=len(R)
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)
    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg * 0.704
    c = concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs = p_nfw.par['rs']/0.704
    A = np.log(1+c)-c/(1+c)
    NFW_pot = NFWPotential(amp=Mg/A*u.Msun, a=rs*u.kpc)

    os = [[r*u.kpc,0*u.km/u.s,vcirc(NFW_pot,r*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg]   for r in R]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot+cdf, method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]
    os = [os[i] for i in valid]
    def test_local(i, m22):
        pot=FDMhalo(m22,Mg)
        orb=Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(m22*1e-22)*u.eV)
        orb.integrate(ts, pot + fdf, method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    def find_m22(i, lo=0.01, hi=100.0, eps=0.01):
        while hi-lo > eps:
            mid = 0.5*(hi+lo)
            if test_local(i, mid):
                hi = mid
            else:
                lo = mid
        return 0.5*(hi+lo)

    m22_c3D[name] = np.array([find_m22(i) for i in range(len(R))])
"""





"""
mc2D=[]
mc3D=[]
for name, m22 in m22_c2D.items():
    if len(m22) > 0:
        mc2D.append(np.min(m22))

for name, m22 in m22_c3D.items():
    if len(m22) > 0:
        mc3D.append(np.min(m22))







#calcul pour r2D et 3D avec excentricité
e=0.5

m22_ce2D = {}
for name, g in groups:
    R = g["R"].to_numpy()
    N_amas=len(R)
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)
    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg * 0.704
    c = concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs = p_nfw.par['rs']/0.704
    A = np.log(1+c)-c/(1+c)
    NFW_pot = NFWPotential(amp=Mg/A*u.Msun, a=rs*u.kpc)

    os = [[r*u.kpc,0*u.km/u.s,np.sqrt(1+e)*vcirc(NFW_pot,r*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg]   for r in R]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot+cdf, method='dop853_c')
        ap = argrelextrema(orb.r(ts), np.greater)[0]
        return np.min(orb.r(ts)[ap])<Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]
    os = [os[i] for i in valid]
    def test_local(i, m22):
        pot=FDMhalo(m22,Mg)
        orb=Orbit([R[i]*u.kpc,0*u.km/u.s,np.sqrt(1+e)*vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(m22*1e-22)*u.eV)
        orb.integrate(ts, pot + fdf, method='dop853_c')
        ap = argrelextrema(orb.r(ts), np.greater)[0]
        return np.min(orb.r(ts)[ap])<Rlimit

    def find_m22(i, lo=0.01, hi=100.0, eps=0.01):
        while hi-lo > eps:
            mid = 0.5*(hi+lo)
            if test_local(i, mid):
                hi = mid
            else:
                lo = mid
        return 0.5*(hi+lo)

    m22_ce2D[name] = np.array([find_m22(i) for i in range(len(R))])


m22_ce3D = {}
for name, g in groups:
    R = g["R"].to_numpy()*2/np.sqrt(3)
    N_amas=len(R)
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)
    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg * 0.704
    c = concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs = p_nfw.par['rs']/0.704
    A = np.log(1+c)-c/(1+c)
    NFW_pot = NFWPotential(amp=Mg/A*u.Msun, a=rs*u.kpc)

    os = [[r*u.kpc,0*u.km/u.s,np.sqrt(1+e)*vcirc(NFW_pot,r*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg]   for r in R]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot+cdf, method='dop853_c')
        ap = argrelextrema(orb.r(ts), np.greater)[0]
        return np.min(orb.r(ts)[ap])<Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]
    os = [os[i] for i in valid]
    def test_local(i, m22):
        pot=FDMhalo(m22,Mg)
        orb=Orbit([R[i]*u.kpc,0*u.km/u.s,np.sqrt(1+e)*vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(m22*1e-22)*u.eV)
        orb.integrate(ts, pot + fdf, method='dop853_c')
        ap = argrelextrema(orb.r(ts), np.greater)[0]
        return np.min(orb.r(ts)[ap])<Rlimit

    def find_m22(i, lo=0.01, hi=100.0, eps=0.01):
        while hi-lo > eps:
            mid = 0.5*(hi+lo)
            if test_local(i, mid):
                hi = mid
            else:
                lo = mid
        return 0.5*(hi+lo)

    m22_ce3D[name] = np.array([find_m22(i) for i in range(len(R))])

mce2D=[]
mce3D=[]
for name, m22 in m22_ce2D.items():
    if len(m22) > 0:
        mce2D.append(np.min(m22))

for name, m22 in m22_ce3D.items():
    if len(m22) > 0:
        mce3D.append(np.min(m22))
        
np.savez("m22_Core_Fornax_test.npz",mc2D=mc2D,mc3D=mc3D,mce2D=mce2D,mce3D=mce3D)
"""

PERSEUS-DWARF-ID10 []
PERSEUS-DWARF-ID100 [np.float64(0.6500000000000001)]
PERSEUS-DWARF-ID1001 []
/tmp/ipykernel_217316/110731608.py:44: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  alpha_sol = fsolve(equation, 10)[0]

/tmp/ipykernel_217316/110731608.py:63: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  alpha_final = fsolve(equation, 10)[0]

PERSEUS-DWARF-ID1002 [np.float64(1.4500000000000006), np.float64(2.5500000000000007)]
PERSEUS-DWARF-ID1005 []
PERSEUS-DWARF-ID1007 []
PERSEUS-DWARF-ID1008 []
PERSEUS-DWARF-ID1009 []
PERSEUS-DWARF-ID1012 [np.float64(1.6000000000000005), np.float64(100.00000000000003)]
PERSEUS-DWARF-ID1013 []
PERSEUS-DWARF-ID1014 [np.float64(1.8000000000000007)]
PERSEUS-DWARF-ID1015 [np.float64(1.6000000000000005)]
PERSEUS-DWARF-ID1016 [np.float64(9.100000000000003)]
PERSEUS-DWARF-ID1017 []
PERSEUS-DWARF-ID1018 [

KeyboardInterrupt: Orbit integration interrupted by CTRL-C (SIGINT)

In [ ]:
#amas qui tombent pas dans cdm
ts = np.linspace(0,10,1001)*u.Gyr
rhm = 0.01 # kpc
halocat = FakeSim()
model = PrebuiltSubhaloModelFactory('behroozi10')
model.populate_mock(halocat)

results1=[]
for name,g in groups:
    logM=g["logMs"].iloc[0]
    Mg=10**model.mean_log_halo_mass(logM)
    Mh=Mg*0.704
    c=concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw=profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs=p_nfw.par['rs']/0.704
    A=np.log(1+c)-c/(1+c)
    NFW_pot=NFWPotential(amp=Mg/A*u.Msun,a=rs*u.kpc)
    for _,row in g.iterrows():
        M=mass_from_mag(row["MAG"])
        orb=Orbit([row["R"]*u.kpc,0*u.km/u.s,vcirc(NFW_pot,row["R"]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        cdf=CDFF(GMs=M*u.Msun,rhm=rhm*u.kpc,dens=NFW_pot)
        orb.integrate(ts,NFW_pot+cdf,method='dop853_c')
        if np.min(orb.r(ts))>=Rlimit:
            results1.append({"MAG":row["MAG"],"R":row["R"],"logMs":logM,"MGC":M,"gal":name})
print(len(results1))

In [ ]:
#amas qui tombe pas CDM

sample=np.random.choice(len(results1),5,replace=False)
m22=100
colors=("green","blue","orange","purple","red")
plt.figure()
for k,i in enumerate(sample):
    cl=results1[i]
    name=cl["gal"]
    M=cl["MGC"]
    R=cl["R"]
    logM=cl["logMs"]
    
    Mg=10**model.mean_log_halo_mass(logM)
    Mh=Mg*0.704
    c=concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw=profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs=p_nfw.par['rs']/0.704
    A=np.log(1+c)-c/(1+c)
    NFW_pot=NFWPotential(amp=Mg/A*u.Msun,a=rs*u.kpc)
    FDM_pot=FDMhalo(m22,Mg)
    
    o_fdm=Orbit([R*u.kpc,0*u.km/u.s,vcirc(FDM_pot,R*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
    o_cdm=Orbit([R*u.kpc,0*u.km/u.s,vcirc(NFW_pot,R*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
    cdf=CDFF(GMs=M*u.Msun,rhm=rhm*u.kpc,dens=NFW_pot)
    df=FDM_DF(GMs=M*u.Msun,rhm=rhm*u.kpc,dens=FDM_pot,m=(m22*1e-22)*u.eV)
    o_cdm.integrate(ts,NFW_pot+cdf,method='dop853_c')
    o_fdm.integrate(ts,FDM_pot+df,method='dop853_c')
    plt.plot(ts.value, o_cdm.r(ts),linestyle="-",color=colors[k % len(colors)],label=f"CDM {name}")
    plt.plot(ts.value, o_fdm.r(ts),linestyle="--",color=colors[k % len(colors)])


    
plt.xlabel("Time [Gyr]", fontsize=17)
plt.axhline(0.2, color='grey', linewidth=1.5, linestyle=':')
plt.ylabel("Radius [kpc]", fontsize=17)
plt.xlim(0, 10)
plt.legend()
plt.show()

In [ ]:
#tombe sous CDM mais tombe pas FDM
plt.figure()
for name, g in groups:
    R = g["R"].to_numpy()
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)

    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg * 0.704

    c = concentration.concentration(Mh, 'vir', 0, model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh, c=c, z=0, mdef='vir')
    rs = p_nfw.par['rs'] / 0.704
    A = np.log(1+c) - c/(1+c)

    NFW_pot = NFWPotential(amp=Mg/A * u.Msun, a=rs * u.kpc)

    os = [[r*u.kpc, 0*u.km/u.s, vcirc(NFW_pot, r*u.kpc)*u.km/u.s, 0*u.kpc, 0*u.km/u.s, 0*u.deg]  for r in R ]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot + cdf, method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]
    os = [os[i] for i in valid]
    
    def test_local(i, m22):
        pot=FDMhalo(m22,Mg)
        orb=Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(m22*1e-22)*u.eV)
        orb.integrate(ts, pot + fdf, method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    def find_m22(i, lo=90, hi=100.0, eps=0.01):
        while hi-lo > eps:
            mid = 0.5*(hi+lo)
            if test_local(i, mid):
                hi = mid
            else:
                lo = mid
        return 0.5*(hi+lo)
    for i in range(len(R)):
            
        if find_m22(i) > 99.9:
            orb = Orbit(os[i])
            cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
            orb.integrate(ts, NFW_pot + cdf, method='dop853_c')
            plt.plot(ts.value, orb.r(ts),linestyle="-",label=name)
            pot=FDMhalo(100,Mg)
            orb=Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
            fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(100*1e-22)*u.eV)
            orb.integrate(ts, pot + fdf, method='dop853_c')
            plt.plot(ts.value, orb.r(ts),linestyle="--")
            
plt.axhline(0.2, color='grey', linewidth=1.5, linestyle=':')
plt.ylabel("Radius [kpc]", fontsize=17)
plt.xlabel("Time [Gyr]", fontsize=17)
plt.xlim(0, 10)
plt.legend()           
plt.show()

In [ ]:
#tombe sous CDM toutes les traj
plt.figure()
for name, g in groups:
    R = g["R"].to_numpy()
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)

    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg * 0.704

    c = concentration.concentration(Mh, 'vir', 0, model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh, c=c, z=0, mdef='vir')
    rs = p_nfw.par['rs'] / 0.704
    A = np.log(1+c) - c/(1+c)

    NFW_pot = NFWPotential(amp=Mg/A * u.Msun, a=rs * u.kpc)

    os = [[r*u.kpc, 0*u.km/u.s, vcirc(NFW_pot, r*u.kpc)*u.km/u.s, 0*u.kpc, 0*u.km/u.s, 0*u.deg]  for r in R ]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot + cdf, method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]
    os = [os[i] for i in valid]
    
    def test_local(i, m22):
        pot=FDMhalo(m22,Mg)
        orb=Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(m22*1e-22)*u.eV)
        orb.integrate(ts, pot + fdf, method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    def find_m22(i, lo=0, hi=100.0, eps=0.1):   #attention epsilon =0.1 dhabitude
        while hi-lo > eps:
            mid = 0.5*(hi+lo)
            if test_local(i, mid):
                hi = mid
            else:
                lo = mid
        return 0.5*(hi+lo)
    
    for i in range(len(R)):
        m=find_m22(i)    
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc, dens=NFW_pot)
        orb.integrate(ts, NFW_pot + cdf, method='dop853_c')
        line, = plt.plot(ts.value, orb.r(ts), linestyle="-")
        color = line.get_color()
        pot=FDMhalo(m,Mg)
        orb=Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun, rhm=rhm*u.kpc,  dens=pot, m=(m*1e-22)*u.eV)
        orb.integrate(ts, pot + fdf, method='dop853_c')
        plt.plot(ts.value, orb.r(ts),linestyle="--",color=color,label=m)
            
plt.axhline(0.2, color='grey', linewidth=1.5, linestyle=':')
plt.ylabel("Radius [kpc]", fontsize=17)
plt.xlabel("Time [Gyr]", fontsize=17)
plt.xlim(0, 10)
plt.legend()           
plt.show()

In [ ]:
#profils densité
m=100
r = np.logspace(-2, 2, 300)
galaxies = ["FORNAX-DWARF-10", "FORNAX-DWARF-4", "FORNAX-DWARF-8"]
M=np.mean([26795212521.146027,47313234474.31487,34691850798.71333])


C2 = concentration.concentration(M * 0.704, 'vir', zgo, model='prada12')
p_nfw = profile_nfw.NFWProfile(M=M * 0.704, c=C2, z=zgo, mdef='vir')
rs = p_nfw.par['rs'] / 0.704
A = np.log(1 + C2) - C2 / (1 + C2)
DMhalo_NFW = NFWPotential(amp=M / A * u.Msun, a=rs * u.kpc)


plt.figure(figsize=(7,5))
rho_nfw = np.array([evaluateDensities(DMhalo_NFW, ri * u.kpc, 0 * u.kpc) * 1e9 for ri in r])
plt.loglog(r, rho_nfw, color="#D9BE98", lw=1.8, ls="-",label="NFW profile")


for i, (name, gal) in enumerate(groups):
    if name not in galaxies:
        continue
    logM = gal["logMs"].iloc[0]
    Mg = 10**model.mean_log_halo_mass(logM)
    pot = FDMhalo(m, Mg)
    rho = np.array([evaluateDensities(pot, ri * u.kpc, 0 * u.kpc) * 1e9 for ri in r])
    plt.loglog(r, rho, alpha=0.8, lw=2.2,  label=rf"{name}")
plt.axvline(0.8,color="grey",linestyle=":",linewidth=1.2)
plt.xlim(0.01,100)
plt.legend()
plt.show()

In [ ]:
Core_fornax = np.load("m22_Core_fornax2.npz")
f_mc2D  = Core_fornax["mc2D"]

vals = []
with open("m22_Fornax_test.txt") as f:
    for line in f:
        vals.append(float(line.split(",")[1]))

print(len(f_mc2D),f_mc2D)

In [8]:
ts = np.linspace(0,10,1001)*u.Gyr
rhm = 0.01*u.kpc
halocat = FakeSim()
model = PrebuiltSubhaloModelFactory('behroozi10')
model.populate_mock(halocat)

df = pd.DataFrame({"ID":IDgal,"R":RGC,"MAG":MAG_GC,"logMs":logMs})
groups = df.groupby("ID")

m22_values = np.arange(0.1,100.0001,0.5)



f = open("m22_Perseus_test_3.txt","a")
start = False

for name, g in groups:

    if not start:
        if name == "PERSEUS-DWARF-ID1209":
            start = True
        continue

    R = g["R"].to_numpy()
    MAG = g["MAG"].to_numpy()
    logM = g["logMs"].iloc[0]
    M = mass_from_mag(MAG)
    Mg = 10**model.mean_log_halo_mass(logM)
    Mh = Mg*0.704
    c = concentration.concentration(Mh,'vir',0,model='prada12')
    p_nfw = profile_nfw.NFWProfile(M=Mh,c=c,z=0,mdef='vir')
    rs = p_nfw.par['rs']/0.704
    A = np.log(1+c)-c/(1+c)
    NFW_pot = NFWPotential(amp=Mg/A*u.Msun,a=rs*u.kpc)
    os = [[r*u.kpc,0*u.km/u.s,vcirc(NFW_pot,r*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg] for r in R]

    def cdm_local(i):
        orb = Orbit(os[i])
        cdf = CDFF(GMs=M[i]*u.Msun,rhm=rhm,dens=NFW_pot)
        orb.integrate(ts,NFW_pot+cdf,method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit

    valid = np.where([cdm_local(i) for i in range(len(R))])[0]
    R = R[valid]
    M = M[valid]

    def test_local(i,m22):
        pot = FDMhalo(m22,Mg)
        orb = Orbit([R[i]*u.kpc,0*u.km/u.s,vcirc(pot,R[i]*u.kpc)*u.km/u.s,0*u.kpc,0*u.km/u.s,0*u.deg])
        fdf = FDM_DF(GMs=M[i]*u.Msun,rhm=rhm,dens=pot,m=(m22*1e-22)*u.eV)
        orb.integrate(ts,pot+fdf,method='dop853_c')
        return np.min(orb.r(ts)) < Rlimit


    def find_m22(i,lo=0.01,hi=100.0,eps=0.1):
        while hi-lo > eps:
            mid = 0.5*(hi+lo)
            if test_local(i,mid):
                hi = mid
            else:
                lo = mid
        return 0.5*(hi+lo)


    def find_m22_full(i):
        res = np.array([test_local(i,m22) for m22 in m22_values])
        v = m22_values[~res]
        return np.nan if len(v)==0 else v[-1]



    vals = []
    for i in range(len(R)):
        m_bis = find_m22(i)
        if m_bis < 5:
            m_full = find_m22_full(i)
            print(name,i,"dichotomie =",m_bis,"all m22=",m_full)
            vals.append(m_full)
        else:
            vals.append(m_bis)
    vals = np.array(vals)

    if len(vals)>0:
        f.write(f"{name},{np.min(vals)}\n")
        f.flush()
     
f.close()


PERSEUS-DWARF-ID1212 0 dichotomie = 1.0352880859374998 all m22= 7.6
PERSEUS-DWARF-ID1220 1 dichotomie = 1.4258740234374998 all m22= 1.1
PERSEUS-DWARF-ID1220 2 dichotomie = 2.1093994140624996 all m22= 1.6
PERSEUS-DWARF-ID1238 0 dichotomie = 1.9141064453125 all m22= 1.6
PERSEUS-DWARF-ID1256 0 dichotomie = 1.1329345703125 all m22= 1.1
PERSEUS-DWARF-ID1259 0 dichotomie = 1.4258740234374998 all m22= 1.1
PERSEUS-DWARF-ID1259 3 dichotomie = 1.9141064453125 all m22= 1.6
PERSEUS-DWARF-ID1263 0 dichotomie = 1.0352880859374998 all m22= 0.6


KeyboardInterrupt: Orbit integration interrupted by CTRL-C (SIGINT)

In [7]:
with open("m22_Perseus_test.txt", "r") as f:
    lignes = f.readlines()

print(len(lignes))


with open("m22_Perseus_test_2.txt", "r") as g:
    lignes = g.readlines()

print(len(lignes))


74
214
